In [1]:
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

In [2]:
URLS = [
    "https://en.wikipedia.org/wiki/2026_Thai_general_election",
    "https://en.wikipedia.org/wiki/2023_Thai_general_election",
    "https://en.wikipedia.org/wiki/2019_Thai_general_election",
    "https://en.wikipedia.org/wiki/2011_Thai_general_election",
    "https://en.wikipedia.org/wiki/2007_Thai_general_election",
    "https://en.wikipedia.org/wiki/2005_Thai_general_election",
    "https://en.wikipedia.org/wiki/2001_Thai_general_election",
]

In [3]:
USER_AGENT = "dsde-final-proj (chatrinza@gmail.com)"  # D-05
REQUEST_DELAY = 1.0  # D-04: polite delay in seconds
OUT_DIR = Path("../data/wikipedia")  # D-09
OUT_FILE = OUT_DIR / "wikipedia_election_results.csv"  # D-10

In [4]:
def fetch_page_html(url: str) -> BeautifulSoup:
    """Fetch full Wikipedia article HTML. Returns BeautifulSoup object.
    Uses lxml parser per research recommendation (faster, more lenient than html.parser).
    """
    headers = {"User-Agent": USER_AGENT}
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "lxml")

In [5]:
def find_results_table(soup: BeautifulSoup):
    """Find the main election results wikitable by locating the 'Results' heading
    and walking forward siblings until the first wikitable is found.

    Anti-pattern avoided: Do NOT use soup.find('table') — the first table on the
    page may be a polling table, image caption, or seat-composition graphic.

    Wikipedia HTML note: Modern Wikipedia wraps h2/h3 inside div.mw-heading elements.
    The wikitable is a sibling of the parent div, not the heading itself.
    This function handles both the old (heading as direct sibling source) and new
    (div.mw-heading wrapper) structure to remain forward-compatible.
    """
    for heading in soup.find_all(["h2", "h3"]):
        if "result" in heading.get_text(strip=True).lower():
            # New Wikipedia structure: heading is wrapped inside div.mw-heading
            parent = heading.parent
            if parent and parent.name == "div" and "mw-heading" in " ".join(parent.get("class", [])):
                for sibling in parent.find_next_siblings():
                    if sibling.name == "table" and "wikitable" in sibling.get("class", []):
                        return sibling
            else:
                # Old structure: wikitable is a direct sibling of the heading
                for sibling in heading.find_next_siblings():
                    if sibling.name == "table" and "wikitable" in sibling.get("class", []):
                        return sibling
    raise ValueError("No results wikitable found on page")

In [6]:
def get_cell_text(cell) -> str:
    """Extract clean text from a td/th, stripping footnote <sup> markers first.

    Anti-pattern avoided: <sup> tags (e.g. [a], [ag], [1]) corrupt numeric parsing
    if not removed before get_text(). Call sup.decompose() universally on all cells.
    """
    for sup in cell.find_all("sup"):
        sup.decompose()
    return cell.get_text(separator=" ", strip=True)

In [7]:
def parse_int(s: str) -> int | None:
    """Remove commas, spaces, and non-digit chars; convert to int.
    Returns None on empty string (e.g., party has 0 seats shown as blank cell).
    Anti-pattern avoided: int("14,438,851") raises ValueError — strip all non-digits first.
    """
    cleaned = re.sub(r"[^\d]", "", s)
    return int(cleaned) if cleaned else None


def parse_float(s: str) -> float | None:
    """Parse percentage string. Returns None on failure.
    Example inputs: "33.7", "33.7%", "–" (em-dash for missing data).
    """
    cleaned = re.sub(r"[^\d.]", "", s)
    return float(cleaned) if cleaned else None

In [8]:
def extract_row_data(cells: list, year: int) -> dict | None:
    """Extract party, seats, votes, vote_pct from a data row's <td> list.
    Returns None for rows that should be skipped (junk/summary rows).

    Column index map:
      Standard layout (2001, 2005, 2007, 2011, 2023, 2026):
        td[0]=swatch, td[1]=party, td[2]=party-list votes, td[3]=party-list %,
        td[4]=PL seats, td[5]=constituency votes, td[6]=constituency %,
        td[7]=constituency seats, td[8]=Total seats, td[9]=+/-
        Minimum expected: 9 cells for standard layout.

      2019 layout (verified at runtime: 7 tds per data row):
        td[0]=swatch, td[1]=party, td[2]=Votes, td[3]=%, td[4]=FPTP seats,
        td[5]=List seats, td[6]=Total seats
        Minimum expected: 7 cells for 2019 layout.

    votes field: party-list votes for standard layout (D-01 says extract total votes;
    party-list votes represent national party support best available column).
    For 2019: single "Votes" column used directly.
    """
    if year == 2019:
        # 2019 requires at least 7 cells for Total seats at index 6
        if len(cells) < 7:
            return None
    else:
        # Standard layout requires at least 9 cells for Total seats at index 8
        if len(cells) < 9:
            return None

    # Party name: td[1], prefer text from <a> link if present
    name_cell = cells[1]
    a_tag = name_cell.find("a")
    party = get_cell_text(a_tag) if a_tag else get_cell_text(name_cell)

    # Skip junk/summary rows — these appear as <td> rows but are not party rows
    SKIP_PATTERNS = [
        "total", "invalid", "blank", "valid votes",
        "registered", "turnout", "none of the above",
        "spoilt", "abstain",
    ]
    if not party or any(p in party.lower() for p in SKIP_PATTERNS):
        return None

    # Skip rows where party name is a pure number/comma string (subtotal/registration rows)
    if re.match(r'^[\d,]+$', party.strip()):
        return None

    # Skip image rows: first cell has colspan > 3 (decorative parliament image row)
    first_cell_colspan = cells[0].get("colspan")
    if first_cell_colspan and int(first_cell_colspan) > 3:
        return None

    if year == 2019:
        # 2019: single Votes column at td[2], %, td[3], Total seats at td[6]
        # Verified at runtime: data rows have 7 tds (swatch + party + votes + % + FPTP + List + Total)
        votes_raw = get_cell_text(cells[2])
        pct_raw = get_cell_text(cells[3])
        seats_raw = get_cell_text(cells[6])
    else:
        # Standard layout: party-list votes at td[2], % at td[3], total seats at td[8]
        votes_raw = get_cell_text(cells[2])
        pct_raw = get_cell_text(cells[3])
        seats_raw = get_cell_text(cells[8])

    votes = parse_int(votes_raw)
    vote_pct = parse_float(pct_raw)
    seats = parse_int(seats_raw)

    # Skip completely empty rows (no votes AND no seats = not a real party row)
    if votes is None and seats is None:
        return None

    return {"party": party, "seats": seats, "votes": votes, "vote_pct": vote_pct}

In [9]:
def parse_results_table(table, year: int) -> list[dict]:
    """Parse a Wikipedia election results wikitable into a list of row dicts.
    Each dict contains: party, seats, votes, vote_pct.

    Anti-pattern avoided: Do NOT build a flat header list from the first <tr>.
    The first row is a group-header row (Party-list, Constituency) with colspan,
    not leaf column names. Use known positional indexes from RESEARCH.md instead.

    Anti-pattern avoided: Do NOT call find_all('tr')[0] for data — may be an
    image row with colspan=10. Use extract_row_data() which skips such rows.
    """
    rows = table.find_all("tr")
    results = []
    for row in rows:
        td_cells = row.find_all("td")
        if not td_cells:
            continue  # Skip header rows (all <th>)
        row_data = extract_row_data(td_cells, year)
        if row_data is not None:
            results.append(row_data)
    return results

In [10]:
# DEBUG: verify 2019 column layout at runtime (Open Question 1 from RESEARCH.md)
# Confirmed: data rows have 7 tds: swatch | party | Votes | % | FPTP | List | Total
# Summary/junk rows (Total, None of the above) have 6 tds and are skipped by len check.
_soup_2019 = fetch_page_html("https://en.wikipedia.org/wiki/2019_Thai_general_election")
_table_2019 = find_results_table(_soup_2019)
_rows_2019 = _table_2019.find_all("tr")
for _row in _rows_2019:
    _tds = _row.find_all("td")
    if len(_tds) >= 7:  # First full data row
        print(f"2019 td count: {len(_tds)}")
        print([get_cell_text(c) for c in _tds])
        break

2019 td count: 7
['', 'Palang Pracharat Party', '8,433,137', '23.34', '97', '19', '116']


In [11]:
# D-06: scrape all 7 elections; D-07: skip 2008 and 2014
ELECTIONS = {
    2026: "https://en.wikipedia.org/wiki/2026_Thai_general_election",
    2023: "https://en.wikipedia.org/wiki/2023_Thai_general_election",
    2019: "https://en.wikipedia.org/wiki/2019_Thai_general_election",
    2011: "https://en.wikipedia.org/wiki/2011_Thai_general_election",
    2007: "https://en.wikipedia.org/wiki/2007_Thai_general_election",
    2005: "https://en.wikipedia.org/wiki/2005_Thai_general_election",
    2001: "https://en.wikipedia.org/wiki/2001_Thai_general_election",
}

In [12]:
all_records = []
MIN_ROWS_PER_YEAR = 10  # Sanity check: each election has 10+ parties

for year, url in ELECTIONS.items():
    print(f"Fetching {year}: {url}")
    soup = fetch_page_html(url)
    table = find_results_table(soup)
    rows = parse_results_table(table, year)

    # Sanity assertion: flag if fewer than 10 party rows scraped (RESEARCH.md Open Question 3)
    if len(rows) < MIN_ROWS_PER_YEAR:
        print(f"  WARNING: Only {len(rows)} rows found for {year} — expected >= {MIN_ROWS_PER_YEAR}")
    else:
        print(f"  OK: {len(rows)} party rows found")

    for row in rows:
        row["year"] = year
        all_records.append(row)

    time.sleep(REQUEST_DELAY)  # D-04: polite delay

print(f"\nTotal records collected: {len(all_records)}")

Fetching 2026: https://en.wikipedia.org/wiki/2026_Thai_general_election


  OK: 57 party rows found


Fetching 2023: https://en.wikipedia.org/wiki/2023_Thai_general_election


  OK: 67 party rows found


Fetching 2019: https://en.wikipedia.org/wiki/2019_Thai_general_election


...

Fetching 2005: https://en.wikipedia.org/wiki/2005_Thai_general_election


  OK: 19 party rows found


Fetching 2001: https://en.wikipedia.org/wiki/2001_Thai_general_election


  OK: 31 party rows found



Total records collected: 310


In [13]:
# D-08: long format — one row per (party, year)
# D-01: columns are party, year, seats, votes, vote_pct
df = pd.DataFrame(all_records, columns=["party", "year", "seats", "votes", "vote_pct"])

print(f"DataFrame shape: {df.shape}")
print(f"Years present: {sorted(df['year'].unique())}")
print(f"\nRows per year:")
print(df.groupby("year").size().sort_index())
print(f"\nSample rows:")
print(df.head(10).to_string())

# Assert all 7 years present
expected_years = {2001, 2005, 2007, 2011, 2019, 2023, 2026}
actual_years = set(df["year"].unique())
missing_years = expected_years - actual_years
if missing_years:
    print(f"WARNING: Missing years: {missing_years}")
else:
    print(f"\nAll 7 years present.")

DataFrame shape: (310, 5)
Years present: [np.int64(2001), np.int64(2005), np.int64(2007), np.int64(2011), np.int64(2019), np.int64(2023), np.int64(2026)]

Rows per year:
year
2001    31
2005    19
2007    27
2011    32
2019    77
2023    67
2026    57
dtype: int64

Sample rows:
                      party  year  seats     votes  vote_pct
0            People's Party  2026    120  11043309     30.56
1         Bhumjaithai Party  2026    192   6468073     17.90
2           Pheu Thai Party  2026     74   5575456     15.43
3            Democrat Party  2026     21   3941928     10.91
4            Economic Party  2026      3   1133055      3.14
5  United Thai Nation Party  2026      2    766078      2.12
6     Pheu Chart Thai Party  2026      2    680256      1.88
7            Kla Tham Party  2026     58    648662      1.79
8       Ruam Jai Thai Party  2026      1    435225      1.20
9          Prachachat Party  2026      5    428848      1.19

All 7 years present.


In [14]:
# D-09: output to data/wikipedia/ (separate from OCR data in data/previous-election/)
# D-10: filename wikipedia_election_results.csv
OUT_DIR.mkdir(parents=True, exist_ok=True)  # WIKI-07: create dir if absent
df["seats"] = df["seats"].astype("Int64")  # Convert float64 → nullable int (avoids '120.0' strings)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df)} rows to {OUT_FILE}")
print(f"Absolute path: {OUT_FILE.resolve()}")

Saved 310 rows to ../data/wikipedia/wikipedia_election_results.csv
Absolute path: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/data/wikipedia/wikipedia_election_results.csv


In [15]:
# Post-export verification
import csv

with open(OUT_FILE, newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows_list = list(reader)

print(f"Header: {header}")
assert header == ["party", "year", "seats", "votes", "vote_pct"], f"Unexpected header: {header}"

years_in_csv = set(int(r[1]) for r in rows_list)
assert years_in_csv == {2001, 2005, 2007, 2011, 2019, 2023, 2026}, f"Missing years: {years_in_csv}"

for year in [2001, 2005, 2007, 2011, 2019, 2023, 2026]:
    count = sum(1 for r in rows_list if int(r[1]) == year)
    assert count >= 10, f"Year {year} has only {count} rows (expected >= 10)"

# Confirm no "Total" junk rows leaked through
party_names = [r[0].lower() for r in rows_list]
assert not any("total" == p.strip() for p in party_names), "Junk 'Total' row found in CSV"

print(f"Verification passed: {len(rows_list)} rows, all 7 years present, no junk rows.")

Header: ['party', 'year', 'seats', 'votes', 'vote_pct']
Verification passed: 310 rows, all 7 years present, no junk rows.
